In [2]:
import pandas as pd

df = pd.read_csv('indian_startup_funding_2020_2025_sample.csv')
print(df.shape)
df.head()

(1100, 8)


,Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
0,Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,19-04-2023
1,Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,28-01-2025
2,Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,14-03-2021
3,FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,11-09-2023
4,Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,31-01-2024


In [3]:
print(df.dtypes)
print()
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

Startup                 object
Industry                object
SubVertical             object
City                    object
Investors               object
InvestmentType          object
InvestmentAmount_USD     int64
Date                    object
dtype: object

Startup                 0
Industry                0
SubVertical             0
City                    0
Investors               0
InvestmentType          0
InvestmentAmount_USD    0
Date                    0
dtype: int64

Duplicate rows: 0


In [4]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Year'] = df['Date'].dt.year
df['Quarter'] = df['Date'].dt.quarter
df.head()

,Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date,Year,Quarter
0,Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,2023-04-19,2023,2
1,Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,2025-01-28,2025,1
2,Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,2021-03-14,2021,1
3,FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,2023-09-11,2023,3
4,Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,2024-01-31,2024,1


In [5]:
df['Investors'] = df['Investors'].str.replace('Tiger Global Management', 'Tiger Global', regex=False)
df.head()

,Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date,Year,Quarter
0,Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,2023-04-19,2023,2
1,Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,2025-01-28,2025,1
2,Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,2021-03-14,2021,1
3,FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,2023-09-11,2023,3
4,Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,2024-01-31,2024,1


In [6]:
df_investors = df.copy()
df_investors['Investors'] = df_investors['Investors'].str.split(',')
df_investors = df_investors.explode('Investors')
df_investors['Investors'] = df_investors['Investors'].str.strip()

print("Original rows:", len(df))
print("After exploding investors:", len(df_investors))
print("Unique investors:", df_investors['Investors'].nunique())

Original rows: 1100
After exploding investors: 1835
Unique investors: 25


In [7]:
df_investors = df_investors.drop_duplicates(subset=['Startup', 'Date', 'Investors'])
print("After removing same-round duplicate investors:", len(df_investors))

After removing same-round duplicate investors: 1829


In [8]:
df_investors['Investors'].value_counts().head(10)

Investors
Tiger Global              136
Y Combinator               83
Mirae Asset                83
Info Edge                  82
Accel                      80
IFC                        78
Sequoia Capital India      77
Kalaari Capital            77
Prosus Ventures            75
Nexus Venture Partners     73
Name: count, dtype: int64

In [9]:
import numpy as np

df['Log_Amount'] = np.log10(df['InvestmentAmount_USD'])
df[['InvestmentAmount_USD', 'Log_Amount']].describe()

,InvestmentAmount_USD,Log_Amount
count,1.100000e+03,1100.000000
mean,2.553295e+07,6.156995
std,7.266655e+07,1.135858
min,5.000000e+03,3.698970
25%,1.697500e+05,5.229808
50%,1.114500e+06,6.047080
75%,1.017250e+07,7.007424
max,5.746760e+08,8.759423


In [10]:
stage_map = {
    'Pre-Seed': 'Early', 'Seed': 'Early', 'Angel': 'Early',
    'Pre-Series A': 'Early', 'Series A': 'Growth', 'Series B': 'Growth',
    'Series C': 'Late', 'Series D': 'Late', 'Growth': 'Late',
    'Private Equity': 'Late', 'Debt': 'Other', 'Bridge': 'Other'
}
df['Stage_Group'] = df['InvestmentType'].map(stage_map)
df['Stage_Group'].value_counts()

Stage_Group
Early     554
Growth    337
Late      149
Other      60
Name: count, dtype: int64

In [11]:
# Round-level file (for funding/amount analysis - one row per deal)
df.to_csv('cleaned_funding_rounds.csv', index=False)

# Investor-level file (for investor analysis - one row per investor per deal)
df_investors.to_csv('cleaned_investors_exploded.csv', index=False)

print("Saved both files successfully")
print("Round-level shape:", df.shape)
print("Investor-level shape:", df_investors.shape)

Saved both files successfully
Round-level shape: (1100, 12)
Investor-level shape: (1829, 10)
